In [ ]:
from datasets import load_dataset
from transformers import TrainingArguments
from trl import SFTTrainer
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from peft import LoraConfig, get_peft_model

In [ ]:


dataset = load_dataset("csv", data_files="news.csv")

def format_data(example):
    return {
        "text": f"""### Instruction:
Generate a news headline.

### Input:
{example['article']}

### Output:
{example['headline']}"""
    }

dataset = dataset.map(format_data)


In [ ]:


model_name = "Qwen/Qwen3-4B-Instruct"   # use 1.8B if low GPU

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    load_in_4bit=True,
    trust_remote_code=True
)


In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj","k_proj","v_proj","o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

tokenized_ds = dataset.map(tokenize, batched=True)


In [ ]:
args = TrainingArguments(
    output_dir="qwen3_headline",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    fp16=True,
    logging_steps=50,
    save_steps=500,
    save_total_limit=2
)

trainer = SFTTrainer(
    model=model,
    train_dataset=tokenized_ds["train"],
    tokenizer=tokenizer,
    args=args
)

trainer.train()
